# YOLO_ChestXray: Master AI Training, Validation & Diagnostic Platform (Plan 1 v4.0)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wupinlai/YOLO_ChestXray/blob/main/notebooks/YOLO_ChestXray_Colab_Training.ipynb)

This notebook implements **Plan 1 v4.0** master plan for YOLOv7 Chest X-Ray pathology detection:
- 🔧 **Colab GPU Runtime Environment & Compatibility Verification** (`reports/environment_report.json`)
- 🔒 **Reproducibility & Non-interactive Execution** (Seed 42, WANDB disabled)
- 🩹 **PyTorch Compatibility Patch** (`weights_only=False`, `reports/patch_report.json`)
- 🔍 **Dataset Integrity & Statistical Profiling** (`reports/dataset_integrity_report.csv`, `reports/dataset_summary.csv`)
- 🚀 **Five-Stage Strict `--resume` Training Workflow** (5 Stages, 10 Epochs / Stage, Total 50 Epochs)
- 🧪 **Post-stage Validation & Metrics Tracking** (`reports/metrics_history.csv`)
- 📊 **Confidence Analysis, IoU Overlap, Failure Taxonomy & Error Annotation**
- 🖼️ **Error Galleries (2x2 Grid) & 10 Random Pre-selected Validation Samples**
- 🔄 **Automatic Runtime Recovery & Checkpoint Search**
- 📄 **Master 16-Page Final PDF Report Compilation** (`reports/final_report/final_report.pdf`)
- 📋 **Project Manifest Generation** (`reports/project_manifest.json`)

## 1. Environment Validation & Reproducibility Setup (Plan 1 v4.0)

In [ ]:
import os, random, sys
import numpy as np
import torch

# 1. Non-Interactive Requirement (WANDB disabled)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

# 2. NumPy Legacy Type Alias Patch
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

# 3. Seed Locking for Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

!nvidia-smi
print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not active. Please switch Runtime to T4/L4 GPU.")

## 2. Google Drive Mounting & Kaggle Credentials

In [ ]:
from google.colab import drive
import os, json, glob

drive.mount('/content/drive')

# Google Drive Structure per Plan 1 v4.0
DRIVE_DIR = '/content/drive/MyDrive/YOLO_ChestXray'
os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/reports", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/final_report", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/final_inference", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/logs", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/releases", exist_ok=True)
os.makedirs(f"{DRIVE_DIR}/datasets", exist_ok=True)

# Load Kaggle Credential from multiple potential sources
kaggle_token = None
kaggle_paths = [
    '/content/drive/MyDrive/Secrets/kaggle_token.txt',
    '/content/drive/MyDrive/Secrets/kaggle.json',
    '/content/YOLO_ChestXray/kaggle_token.txt',
    'kaggle_token.txt'
]
for kp in kaggle_paths:
    if os.path.exists(kp):
        try:
            with open(kp, 'r') as f:
                content = f.read().strip()
                if content.startswith('{'):
                    data = json.loads(content)
                    kaggle_token = data.get('key') or data.get('token')
                else:
                    kaggle_token = content
            print(f"Loaded Kaggle token from {kp}!")
            break
        except Exception:
            pass

if not kaggle_token:
    try:
        from google.colab import userdata
        kaggle_token = userdata.get('KAGGLE_KEY') or userdata.get('KAGGLE_API_TOKEN')
        if kaggle_token:
            print("Loaded Kaggle token from Colab Secrets!")
    except Exception:
        pass

# Fallback default token if none provided
if not kaggle_token:
    kaggle_token = 'KGAT_13ad811ce4a53473588e656be1abd413'

os.environ['KAGGLE_KEY'] = kaggle_token
os.environ['KAGGLE_API_TOKEN'] = kaggle_token
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': 'kaggle_user', 'key': kaggle_token, 'token': kaggle_token}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
print("Kaggle API credential configured.")

## 3. Clone Repository & Setup YOLOv7 Compatibility Environment (Plan 1 v4.0)

In [ ]:
%cd /content
!git clone https://github.com/wupinlai/YOLO_ChestXray.git
%cd /content/YOLO_ChestXray

# Clone YOLOv7 official base
!git clone https://github.com/WongKinYiu/yolov7.git yolov7_repo
!cp -r yolov7_repo/* ./ || true

# Per Plan 1 v4.0: Install only required missing packages without overwriting PyTorch
!pip install -q seaborn tensorboard pycocotools pypdf

# Apply YOLOv7 PyTorch & torch.load Compatibility Patch Layer
!python scripts/fix_yolov7_env.py ./ reports/

# Download official pretrained YOLOv7 weights
!wget -nc https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

## 4. Download Kaggle Dataset & Integrity Check (Plan 1 v4.0)

In [ ]:
import os, glob, shutil
from pathlib import Path

os.makedirs('datasets/chestxray8', exist_ok=True)
zip_name = 'yolo-annotated-chestxray-8-object-detection.zip'
drive_zip = f"{DRIVE_DIR}/datasets/{zip_name}"
local_zip = f"datasets/{zip_name}"

# 1. Check if cached in Google Drive first for lightning fast loading
if os.path.exists(drive_zip):
    print(f"Found cached dataset in Google Drive: {drive_zip}. Copying...")
    !cp {drive_zip} datasets/
elif not os.path.exists(local_zip):
    print("Downloading dataset via Kaggle API...")
    !kaggle datasets download -d spritan1/yolo-annotated-chestxray-8-object-detection -p datasets/
    if os.path.exists(local_zip):
        os.makedirs(f"{DRIVE_DIR}/datasets", exist_ok=True)
        !cp {local_zip} {DRIVE_DIR}/datasets/ || true

# 2. Unzip dataset
if os.path.exists(local_zip):
    !unzip -q -o {local_zip} -d datasets/chestxray8/
    print("Dataset unzipped successfully.")
else:
    print("Checking for extracted or alternative zip files...")
    for z in glob.glob('datasets/*.zip'):
        !unzip -q -o {z} -d datasets/chestxray8/

# 3. Auto-Standardize Folder Hierarchy (handles any nested unzipped structure)
from scripts.dataset_utils import prepare_and_standardize_dataset, check_dataset_integrity
t_img, t_lbl, v_img, v_lbl = prepare_and_standardize_dataset('datasets/chestxray8')

# 4. Run Dataset Integrity Check
df_integrity = check_dataset_integrity(
    t_img, t_lbl, v_img, v_lbl,
    'reports/dataset_integrity_report.csv'
)
print("--- Dataset Integrity Summary ---")
print(df_integrity.head(10))
!cp reports/dataset_integrity_report.csv {DRIVE_DIR}/reports/ || true

## 5. Dataset Configuration & Statistical Analysis

In [ ]:
import yaml
from scripts.dataset_utils import run_dataset_analysis

config = {
    'train': os.path.abspath(t_img),
    'val': os.path.abspath(v_img),
    'nc': 14,
    'names': [
        'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass',
        'Nodule', 'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema',
        'Emphysema', 'Fibrosis', 'Pleural_Thickening', 'Hernia'
    ]
}
os.makedirs('configs', exist_ok=True)
with open('configs/chestxray.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f"Generated dataset config at configs/chestxray.yaml with train={t_img}, val={v_img}")

# Run dataset statistical analysis
run_dataset_analysis(
    'configs/chestxray.yaml',
    t_img, t_lbl, v_img, v_lbl,
    'reports/dataset_analysis'
)
!cp -r reports/dataset_analysis/* {DRIVE_DIR}/reports/ || true

## 6. Pre-training: Select 10 Random Validation Samples (Plan 1 v4.0)

In [ ]:
!python scripts/inference.py --action sample --val-dir {v_img} --output-dir reports/final_inference --num-samples 10
!cp -r reports/final_inference/* {DRIVE_DIR}/final_inference/ || true

## 7. Stage-wise Training Pipeline (Strict Plan 1 v4.0 `--resume` Policy)

In [ ]:
# Stage 1: Epochs 1 ~ 10
!python scripts/train.py --weights yolov7.pt --stage 1 --epochs 10 --batch-size 16 --name stage_1
!cp checkpoints/checkpoint_10.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 2: Epochs 11 ~ 20 (Resume from checkpoint_10.pt)
!python scripts/train.py --resume checkpoints/checkpoint_10.pt --stage 2 --epochs 20 --name stage_2
!cp checkpoints/checkpoint_20.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 3: Epochs 21 ~ 30 (Resume from checkpoint_20.pt)
!python scripts/train.py --resume checkpoints/checkpoint_20.pt --stage 3 --epochs 30 --name stage_3
!cp checkpoints/checkpoint_30.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 4: Epochs 31 ~ 40 (Resume from checkpoint_30.pt)
!python scripts/train.py --resume checkpoints/checkpoint_30.pt --stage 4 --epochs 40 --name stage_4
!cp checkpoints/checkpoint_40.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

In [ ]:
# Stage 5: Epochs 41 ~ 50 (Resume from checkpoint_40.pt)
!python scripts/train.py --resume checkpoints/checkpoint_40.pt --stage 5 --epochs 50 --name stage_5
!cp checkpoints/checkpoint_50.pt {DRIVE_DIR}/checkpoints/ || true
!cp checkpoints/best_model.pt {DRIVE_DIR}/checkpoints/ || true
!cp reports/metrics_history.csv {DRIVE_DIR}/reports/ || true

## 8. Validation on Best Model Checkpoint

In [ ]:
# Check for available checkpoints across Drive and local before validation
import os, shutil
os.makedirs('checkpoints', exist_ok=True)
if not os.path.exists('checkpoints/best_model.pt'):
    for ckpt in [f"{DRIVE_DIR}/checkpoints/best_model.pt", f"{DRIVE_DIR}/checkpoints/checkpoint_50.pt", 'yolov7.pt']:
        if os.path.exists(ckpt):
            shutil.copy(ckpt, 'checkpoints/best_model.pt')
            print(f"Loaded checkpoint for validation: {ckpt}")
            break

!python scripts/validate.py --weights checkpoints/best_model.pt --data configs/chestxray.yaml --name val_results
!cp reports/*.png {DRIVE_DIR}/reports/ || true
!cp reports/*.csv {DRIVE_DIR}/reports/ || true

## 9. Comprehensive Plan 1 v4.0 Post-Processing, Diagnostics & 16-Page Master PDF Report

In [ ]:
# 1. Force update to latest GitHub main version (wiping any stale temporary files in Colab)
%cd /content/YOLO_ChestXray
!git fetch origin main
!git reset --hard origin/main

# 2. Run Plan 1 v4.0 post-processing, case annotations, and 16-page master PDF
!PYTHONPATH=/content/YOLO_ChestXray python scripts/run_plan1_pipeline.py

# 3. Sync all reports and final PDF to Google Drive
!cp -r reports/* {DRIVE_DIR}/reports/ || true
!cp reports/final_report/final_report.pdf {DRIVE_DIR}/final_report/final_report.pdf || true
print("All Plan 1 v4.0 deliverables and 16-page master PDF synced to Google Drive!")

# 4. Trigger automatic browser download of the PDF report
try:
    from google.colab import files
    if os.path.exists('reports/final_report/final_report.pdf'):
        files.download('reports/final_report/final_report.pdf')
        print("Download of final_report.pdf initiated!")
except Exception as e:
    print(f"PDF is available at {DRIVE_DIR}/final_report/final_report.pdf")


## 10. Display Visual Results (Inference & Error Galleries)

In [ ]:
from IPython.display import Image, display
import os

print('=== Final Multi-Sample Inference Result (Sample 1) ===')
if os.path.exists('reports/final_inference/result_01.jpg'):
    display(Image(filename='reports/final_inference/result_01.jpg', width=600))

print('=== [Page 10] Best Cases: Top High-Accuracy Detections ===')
if os.path.exists('reports/top5_detections.jpg'):
    display(Image(filename='reports/top5_detections.jpg', width=700))

print('=== [Page 11] Worst Cases and Boundary Discrepancies ===')
if os.path.exists('reports/worst_iou_cases.jpg'):
    display(Image(filename='reports/worst_iou_cases.jpg', width=700))

print('=== [Page 12] False Positive Error Gallery ===')
if os.path.exists('reports/fp_gallery.jpg'):
    display(Image(filename='reports/fp_gallery.jpg', width=700))

print('=== [Page 13] False Negative Error Gallery ===')
if os.path.exists('reports/fn_gallery.jpg'):
    display(Image(filename='reports/fn_gallery.jpg', width=700))

print('=== [Page 14] Misclassification Cases Gallery ===')
if os.path.exists('reports/misclassification_gallery.jpg'):
    display(Image(filename='reports/misclassification_gallery.jpg', width=700))

print('=== [Page 15] Low IoU Boundary Cases Gallery ===')
if os.path.exists('reports/low_iou_gallery.jpg'):
    display(Image(filename='reports/low_iou_gallery.jpg', width=700))

print('=== Confidence Threshold Sweep Analysis (0.1 to 0.9) ===')
if os.path.exists('reports/confidence_threshold_analysis.png'):
    display(Image(filename='reports/confidence_threshold_analysis.png', width=600))


## 11. Runtime Recovery (Auto Resume on Disconnect)

In [ ]:
import os

# Check Google Drive and local checkpoints for latest checkpoint
latest_ckpt = None
for ep in [50, 40, 30, 20, 10]:
    drive_ckpt = f"{DRIVE_DIR}/checkpoints/checkpoint_{ep}.pt"
    local_ckpt = f"checkpoints/checkpoint_{ep}.pt"
    if os.path.exists(drive_ckpt):
        os.makedirs('checkpoints', exist_ok=True)
        !cp {drive_ckpt} checkpoints/
        latest_ckpt = local_ckpt
        next_stage = (ep // 10) + 1
        next_epochs = ep + 10
        print(f"Found latest checkpoint: {drive_ckpt} (Epoch {ep}). Ready to resume Stage {next_stage} (Target {next_epochs} Epochs).")
        break

if latest_ckpt and next_stage <= 5:
    print(f"Resuming training from {latest_ckpt}...")
    !python scripts/train.py --resume {latest_ckpt} --stage {next_stage} --epochs {next_epochs} --name stage_{next_stage}
else:
    print("No prior stage checkpoints found or all 5 stages completed.")

## 12. Package Release Assets for GitHub Release (Plan 1 v4.0)

In [ ]:
!zip -r {DRIVE_DIR}/releases/Release_50_Plan1_v4.0_assets.zip \
  checkpoints/checkpoint_50.pt \
  checkpoints/best_model.pt \
  reports/final_report/final_report.pdf \
  reports/environment_report.json \
  reports/patch_report.json \
  reports/dataset_integrity_report.csv \
  reports/project_manifest.json \
  reports/*.png \
  reports/*.csv

print(f"Plan 1 v4.0 Release bundle created at {DRIVE_DIR}/releases/Release_50_Plan1_v4.0_assets.zip")